# Week 1 Tuesday — Telco Churn EDA

Data source: Kaggle `blastchar/telco-customer-churn`, uploaded to `s3://beant-mlops-portfolio-666258711441/raw/`.

In [1]:
import boto3
import pandas as pd

BUCKET = "beant-mlops-portfolio-666258711441"
KEY = "raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=BUCKET, Key=KEY)
df = pd.read_csv(obj["Body"])
df.shape

(7043, 21)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reload (or reuse df from Monday)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Only 11 nulls (new customers, tenure=0) - safe to impute as 0 rather than drop
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# customerID is a unique identifier, not a feature - drop it
df = df.drop(columns=["customerID"])

# Target: convert Yes/No to 1/0
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn"])

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]


done


# Sanity-Check the split:

In [3]:
X["SeniorCitizen"] = X["SeniorCitizen"].astype(str)  # it's 0/1 int but semantically categorical
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print(numeric_cols)
print(categorical_cols)

['tenure', 'MonthlyCharges', 'TotalCharges']
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


# Column Transformer

In [4]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
print(X_train_t.shape, X_test_t.shape)

(5634, 46) (1409, 46)
